# 09b — Alignement des images sur le cerveau de référence (MNI) — 112 participants

**Pourquoi.** Le notebook 09 étirait chaque image en 64×64×64 sans l'aligner : un même point du cube ne
correspondait pas à la même zone du cerveau d'un patient à l'autre (6 tailles d'IRM différentes, crâne
inclus). Un CNN ou une PCA ne peut pas apprendre « où regarder » dans ces conditions.

**Ce que fait ce notebook** (le code est dans `src/alignement_mni.py`) :

- **IRM** : recalage affine sur le modèle MNI avec crâne → retrait du crâne → correction du biais
  d'intensité → normalisation (z-score) dans le cerveau.
- **PET** : recalé sur l'IRM *du même patient*, puis déplacé dans le MNI avec la même transformation.
- Les deux : recadrage commun puis cube **64×64×64**, mêmes axes pour tous
  (axe 0 = gauche→droite, axe 1 = arrière→avant, axe 2 = bas→haut).
- **Mesures par région** (hippocampe, amygdale, ventricules, temporal, pariétal postérieur, cingulaire postérieur,
  frontal) : part de liquide / matière grise pour l'IRM, activité relative pour le PET.
- **Contrôle qualité** pour chaque patient, avec nouvel essai automatique si le recalage est mauvais.

Sorties dans un nouveau dossier `features_image_mni_<date>/` : `volumes_mni/*.npy`, `features_regions.csv`,
`qc_alignement.csv`, et des figures de contrôle. Le calcul est **reprenable** : on peut l'interrompre et le relancer.

In [ ]:
%pip install nibabel antspyx nilearn

In [ ]:
from pathlib import Path
import sys, time, traceback
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(r"..\configs")
sys.path.append(r"..\src")
from config_projet import C
import alignement_mni as A

modele = A.ModeleMNI(C.DATA / "08_Modele_MNI")
print("Modele MNI charge. Regions :", {k: int(v.sum()) for k, v in modele.regions.items()})

base = pd.read_csv(C.BASE_112, dtype={"PTID": "string"})
base["PTID"] = base["PTID"].str.strip().str.upper()


def chemin_local(p):
    """Les chemins du manifeste ont ete ecrits sur le disque E: : on garde la partie apres 'data\\'."""
    return C.DATA / Path(str(p).split("data\\", 1)[1].replace("\\", "/"))


# Dossier de sortie : on reprend le dernier s'il existe (calcul reprenable).
existants = sorted(C.COHORTE.glob("features_image_mni_*"))
SORTIE = existants[-1] if existants else C.sortie("features_image_mni")
DOSSIER = SORTIE / "volumes_mni"
DOSSIER.mkdir(exist_ok=True)
print("Sortie :", SORTIE.name, "| participants :", len(base))

## 1. Alignement des 112 participants (≈ 25 à 50 s chacun)

In [ ]:
journal = SORTIE / "features_regions.csv"
lignes = pd.read_csv(journal, dtype={"PTID": "string"}).to_dict("records") if journal.exists() else []
faits = {l["PTID"] for l in lignes}
erreurs = []
debut = time.time()

for n, r in enumerate(base.itertuples(index=False), start=1):
    f_irm, f_pet = DOSSIER / f"{r.PTID}_IRM.npy", DOSSIER / f"{r.PTID}_PET.npy"
    if r.PTID in faits and f_irm.exists() and f_pet.exists():
        continue
    try:
        irm, pet, mesures, qc = A.traiter_patient(modele, chemin_local(r.chemin_irm), chemin_local(r.chemin_pet))
        np.save(f_irm, irm)
        np.save(f_pet, pet)
        lignes = [l for l in lignes if l["PTID"] != r.PTID]
        lignes.append({"PTID": r.PTID, **mesures, **qc})
        pd.DataFrame(lignes).to_csv(journal, index=False, encoding="utf-8-sig")
    except Exception as e:
        erreurs.append({"PTID": r.PTID, "erreur": f"{type(e).__name__}: {e}"})
        traceback.print_exc()
    if n % 5 == 0:
        print(f"{n}/{len(base)} | {len(lignes)} faits | {(time.time() - debut) / 60:.1f} min", flush=True)

print("\nTermine :", len(lignes), "participants alignes |", len(erreurs), "erreurs")
if erreurs:
    display(pd.DataFrame(erreurs))
print("Volumes .npy :", len(list(DOSSIER.glob("*.npy"))), "(attendu : 224)")

## 2. Contrôle qualité

- `IRM_info_mutuelle` : plus c'est **négatif**, mieux l'IRM est alignée. Valeur habituelle : environ −0,70 ;
  **alerte si supérieure à −0,60**.
- `PET_corr_matiere_grise` : le PET aligné est plus actif dans la matière grise du modèle. **Bon : > 0,2** ;
  un PET mal aligné donne environ 0.
- `*_essais` : nombre de tentatives nécessaires (1 = tout de suite réussi).

In [ ]:
qc = pd.read_csv(journal, dtype={"PTID": "string"})
cols = ["IRM_info_mutuelle", "IRM_essais", "PET_corr_matiere_grise", "PET_essais"]
display(qc[cols].describe().round(3))

irm_suspects = qc.loc[qc["IRM_info_mutuelle"] > A.SEUIL_MI, "PTID"].tolist()
pet_suspects = qc.loc[qc["PET_corr_matiere_grise"] < A.SEUIL_PET, "PTID"].tolist()
print("IRM suspectes (info. mutuelle >", A.SEUIL_MI, ") :", len(irm_suspects))
print("PET suspects (correlation <", A.SEUIL_PET, ") :", len(pet_suspects))
print("Patients ayant eu besoin de plusieurs essais (IRM / PET) :",
      int((qc["IRM_essais"] > 1).sum()), "/", int((qc["PET_essais"] > 1).sum()))
qc.to_csv(SORTIE / "qc_alignement.csv", index=False, encoding="utf-8-sig")

## 3. Contrôle visuel : le cerveau moyen et l'écart-type doivent être nets et centrés

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

ptids = qc["PTID"].tolist()
IRM = np.stack([np.load(DOSSIER / f"{p}_IRM.npy") for p in ptids])
PET = np.stack([np.load(DOSSIER / f"{p}_PET.npy") for p in ptids])
print("Piles :", IRM.shape, PET.shape)

centre = [32, 32, 32]
fig, ax = plt.subplots(4, 3, figsize=(9, 12), constrained_layout=True)
for ligne, (nom, pile, f) in enumerate([("IRM moyenne", IRM, np.mean), ("IRM ecart-type", IRM, np.std),
                                         ("PET moyen", PET, np.mean), ("PET ecart-type", PET, np.std)]):
    vol = f(pile, axis=0)
    coupes = [vol[centre[0], :, :], vol[:, centre[1], :], vol[:, :, centre[2]]]
    for col, (c, titre) in enumerate(zip(coupes, ["sagittale", "coronale", "axiale"])):
        ax[ligne, col].imshow(np.rot90(c), cmap="gray")
        ax[ligne, col].set_title(f"{nom} - {titre}", fontsize=9)
        ax[ligne, col].axis("off")
fig.savefig(SORTIE / "controle_cerveau_moyen.png", dpi=110)
plt.show()

In [ ]:
# Montage : une coupe axiale de chaque patient (IRM puis PET). Les cerveaux doivent tous se ressembler.
def montage(pile, nom, vmin, vmax):
    n = len(pile)
    col = 14
    lig = int(np.ceil(n / col))
    fig, ax = plt.subplots(lig, col, figsize=(col * 1.1, lig * 1.1), constrained_layout=True)
    for i, a in enumerate(ax.ravel()):
        a.axis("off")
        if i < n:
            a.imshow(np.rot90(pile[i][:, :, 34]), cmap="gray", vmin=vmin, vmax=vmax)
    fig.suptitle(nom)
    fig.savefig(SORTIE / f"montage_{nom}.png", dpi=90)
    plt.show()

montage(IRM, "IRM", -1.5, 2.5)
montage(PET, "PET", 0, 2.5)

In [ ]:
# Les cas suspects, s'il y en a : coupe axiale IRM + PET.
suspects = sorted(set(irm_suspects) | set(pet_suspects))
print("Cas a regarder :", len(suspects))
if suspects:
    fig, ax = plt.subplots(len(suspects), 2, figsize=(5, 2.4 * len(suspects)), constrained_layout=True, squeeze=False)
    for i, p in enumerate(suspects):
        k = ptids.index(p)
        ax[i, 0].imshow(np.rot90(IRM[k][:, :, 34]), cmap="gray", vmin=-1.5, vmax=2.5)
        ax[i, 1].imshow(np.rot90(PET[k][:, :, 34]), cmap="gray", vmin=0, vmax=2.5)
        ax[i, 0].set_title(f"IRM #{k}", fontsize=8)
        ax[i, 1].set_title(f"PET #{k}", fontsize=8)
        ax[i, 0].axis("off"); ax[i, 1].axis("off")
    plt.show()